# Создание единого датасета из трех подготовленных датасетов

- Спотовые котировки BTC (Kaggle) - Bitcoin_history_data_final.csv
- Фьючерсы CME (Yahoo Finance) - cme_btc_futures_daily_final.csv
- Потоки капитала спотовых BTC ETF (Farside Investors) - bitcoin_etf_final.csv

In [1]:
import pandas as pd

# 1. Загружаем три подготовленных датасета
df_spot = pd.read_csv("Bitcoin_history_data_final.csv")
df_cme = pd.read_csv("cme_btc_futures_daily_final.csv")
df_etf = pd.read_csv("bitcoin_etf_final.csv")

# 2. Приводим столбец Date к единому типу datetime во всех таблицах
# Это критически важно для корректного сопоставления строк при объединении
df_spot['Date'] = pd.to_datetime(df_spot['Date'])
df_cme['Date'] = pd.to_datetime(df_cme['Date'])
df_etf['Date'] = pd.to_datetime(df_etf['Date'])

# 3. Первое объединение: Спот + Фьючерсы CME
# how='left' сохраняет все даты из базового спотового датасета
df_merged = pd.merge(df_spot, df_cme, on='Date', how='left')

# 4. Второе объединение: Результат + Потоки ETF
df_final = pd.merge(df_merged, df_etf, on='Date', how='left')

# 5. Сортируем итоговый датасет по дате для хронологического порядка
df_final = df_final.sort_values('Date').reset_index(drop=True)

# 6. Сохраняем итоговый сводный датасет
final_merged_file = "btc_comprehensive_analysis.csv"
df_final.to_csv(final_merged_file, index=False, encoding='utf-8')

print(f"✅ Датасеты успешно объединены и сохранены как: {final_merged_file}")
print(f"Итоговый размер: {df_final.shape[0]} строк, {df_final.shape[1]} столбцов")

print("\nПроверка наличия пропусков (NaN) по ключевым новым столбцам:")
# Считаем количество пропусков в столбцах, которых не было в базовом датасете
cols_to_check = ['F_Close', 'ETF_Total_Net_Flow_mln_usd', 'ETF_Cumulative_Flow_mln_usd']
for col in cols_to_check:
    if col in df_final.columns:
        na_count = df_final[col].isna().sum()
        print(f"- {col}: {na_count} пропусков (это ожидаемо для периодов до запуска инструмента)")

print("\nПервые 5 строк объединенного датасета (для визуального контроля):")
display(df_final.head())

print("\nПоследние 5 строк объединенного датасета (проверка актуальности):")
display(df_final.tail())

✅ Датасеты успешно объединены и сохранены как: btc_comprehensive_analysis.csv
Итоговый размер: 4175 строк, 16 столбцов

Проверка наличия пропусков (NaN) по ключевым новым столбцам:
- F_Close: 2118 пропусков (это ожидаемо для периодов до запуска инструмента)
- ETF_Total_Net_Flow_mln_usd: 3630 пропусков (это ожидаемо для периодов до запуска инструмента)
- ETF_Cumulative_Flow_mln_usd: 3630 пропусков (это ожидаемо для периодов до запуска инструмента)

Первые 5 строк объединенного датасета (для визуального контроля):


,Date,Close,High,Low,Open,market_era,halving_cycle,Volume_mln,F_Close,F_High,F_Low,F_Open,F_Volume_Contracts,F_Volume_USD_Mln,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
0,2014-09-17,457.3,468.2,452.4,465.9,Pre_Institutional,1,21.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-09-18,424.4,456.9,413.1,456.9,Pre_Institutional,1,34.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-09-19,394.8,427.8,384.5,424.1,Pre_Institutional,1,37.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-09-20,408.9,423.3,389.9,394.7,Pre_Institutional,1,36.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-09-21,398.8,412.4,393.2,408.1,Pre_Institutional,1,26.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Последние 5 строк объединенного датасета (проверка актуальности):


,Date,Close,High,Low,Open,market_era,halving_cycle,Volume_mln,F_Close,F_High,F_Low,F_Open,F_Volume_Contracts,F_Volume_USD_Mln,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
4170,2026-02-16,68843.2,70067.2,67301.6,68782.4,ETF_Era,4,33618.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4171,2026-02-17,67494.2,69201.9,66615.3,68843.1,ETF_Era,4,34866.9,67865.0,70170.0,66635.0,69120.0,10389.0,3525.2,-104.9,55320.1
4172,2026-02-18,66425.3,68434.4,65845.9,67488.0,ETF_Era,4,33094.3,66330.0,68570.0,65895.0,67625.0,7503.0,2488.4,-133.3,55186.8
4173,2026-02-19,66957.5,67277.1,65637.4,66425.6,ETF_Era,4,31493.0,67205.0,67400.0,65680.0,66425.0,7266.0,2441.6,-165.8,55021.0
4174,2026-02-20,68005.4,68269.0,66452.5,66958.6,ETF_Era,4,47507.9,67825.0,68450.0,66565.0,66970.0,10745.0,3643.9,88.1,55109.1


In [2]:
import pandas as pd

# 1. Загружаем объединенный датасет
df_merged = pd.read_csv("btc_comprehensive_analysis.csv")

print("=== 1. Общая информация и типы данных ===")
# info() покажет, не превратились ли числовые столбцы в object из-за NaN или ошибок слияния
df_merged.info()

print("\n=== 2. Проверка пропусков (NaN) ===")
# Считаем процент пропусков по каждому столбцу
missing_stats = df_merged.isna().sum() / len(df_merged) * 100
# Показываем только те столбцы, где есть пропуски, отсортированные по убыванию
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)
print("Столбцы с пропусками (% от общего числа строк):")
print(missing_stats.round(2))

print("\n=== 3. Проверка на дубликаты по дате ===")
# Убедимся, что каждая дата встречается только один раз
duplicates_count = df_merged['Date'].duplicated().sum()
print(f"Найдено дубликатов дат: {duplicates_count}")

print("\n=== 4. Визуальный контроль границ данных ===")
print("Первые 3 строки (должны быть NaN в F_ и ETF_ столбцах):")
display(df_merged.head(3))

print("Строки за 2024 год (должны быть заполнены все столбцы):")
# Фильтруем пример за 2024 год для проверки
display(df_merged[df_merged['Date'] >= '2024-01-15'].head(3))

=== 1. Общая информация и типы данных ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4175 entries, 0 to 4174
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Date                         4175 non-null   object 
 1   Close                        4175 non-null   float64
 2   High                         4175 non-null   float64
 3   Low                          4175 non-null   float64
 4   Open                         4175 non-null   float64
 5   market_era                   4175 non-null   object 
 6   halving_cycle                4175 non-null   int64  
 7   Volume_mln                   4175 non-null   float64
 8   F_Close                      2057 non-null   float64
 9   F_High                       2057 non-null   float64
 10  F_Low                        2057 non-null   float64
 11  F_Open                       2057 non-null   float64
 12  F_Volume_Contracts           2057 

,Date,Close,High,Low,Open,market_era,halving_cycle,Volume_mln,F_Close,F_High,F_Low,F_Open,F_Volume_Contracts,F_Volume_USD_Mln,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
0,2014-09-17,457.3,468.2,452.4,465.9,Pre_Institutional,1,21.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-09-18,424.4,456.9,413.1,456.9,Pre_Institutional,1,34.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-09-19,394.8,427.8,384.5,424.1,Pre_Institutional,1,37.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Строки за 2024 год (должны быть заполнены все столбцы):


,Date,Close,High,Low,Open,market_era,halving_cycle,Volume_mln,F_Close,F_High,F_Low,F_Open,F_Volume_Contracts,F_Volume_USD_Mln,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
3407,2024-01-15,42512.0,43319.7,41705.4,41715.1,ETF_Era,3,22320.2,NaN,NaN,NaN,NaN,NaN,NaN,0.0,858.3
3408,2024-01-16,43154.9,43566.3,42086.0,42499.3,ETF_Era,3,24062.9,43295.0,43675.0,41705.0,42365.0,14369.0,3110.5,-52.7,805.6
3409,2024-01-17,42742.7,43189.9,42189.3,43132.1,ETF_Era,3,20851.2,42810.0,43405.0,42215.0,43370.0,7598.0,1626.4,453.8,1259.4


In [5]:
import pandas as pd

# 1. Загружаем объединенный датасет
df_final = pd.read_csv("btc_comprehensive_analysis.csv")

# 2. Переводим столбец Date в формат datetime64[ns]
# Теперь pandas будет понимать, что это даты, а не просто текст
df_final['Date'] = pd.to_datetime(df_final['Date'])

# 3. Сортируем по дате на всякий случай (чтобы хронология была строгой)
df_final = df_final.sort_values('Date').reset_index(drop=True)

# 4. Пересохраняем файл с исправленным типом данных
final_file = "btc_comprehensive_analysis.csv"
df_final.to_csv(final_file, index=False, encoding='utf-8')

# 5. Проверяем, что тип изменился
print("✅ Тип столбца Date успешно изменен.")
print(f"Новый тип данных для Date: {df_final['Date'].dtype}")

print("\nПервые 3 строки (визуальный контроль):")
display(df_final.head(3))

✅ Тип столбца Date успешно изменен.
Новый тип данных для Date: datetime64[ns]

Первые 3 строки (визуальный контроль):


,Date,Close,High,Low,Open,market_era,halving_cycle,Volume_mln,F_Close,F_High,F_Low,F_Open,F_Volume_Contracts,F_Volume_USD_Mln,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
0,2014-09-17,457.3,468.2,452.4,465.9,Pre_Institutional,1,21.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-09-18,424.4,456.9,413.1,456.9,Pre_Institutional,1,34.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-09-19,394.8,427.8,384.5,424.1,Pre_Institutional,1,37.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
import pandas as pd

# 1. Загружаем данные из CSV-файла в DataFrame (таблицу в памяти)
# Мы сохраняем результат в переменную df (стандартное сокращение от DataFrame)
df = pd.read_csv("btc_comprehensive_analysis.csv")

# 2. Смотрим размер таблицы: (количество строк, количество столбцов)
print("=== 1. Размер таблицы ===")
print(f"Строк: {df.shape[0]}, Столбцов: {df.shape[1]}\n")

# 3. Выводим первые 5 строк для визуальной оценки
print("=== 2. Первые 5 строк данных ===")
display(df.head())

# 4. Выводим список всех столбцов в виде простого списка
print("=== 3. Список столбцов ===")
print(df.columns.tolist(), "\n")

# 5. Смотрим типы данных для каждого столбца
print("=== 4. Типы данных (dtypes) ===")
print(df.dtypes, "\n")

# 6. Получаем сводную информацию о таблице (самый важный этап осмотра)
print("=== 5. Сводная информация (info) ===")
df.info()

=== 1. Размер таблицы ===
Строк: 4175, Столбцов: 16

=== 2. Первые 5 строк данных ===


,Date,Close,High,Low,Open,market_era,halving_cycle,Volume_mln,F_Close,F_High,F_Low,F_Open,F_Volume_Contracts,F_Volume_USD_Mln,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
0,2014-09-17,457.3,468.2,452.4,465.9,Pre_Institutional,1,21.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-09-18,424.4,456.9,413.1,456.9,Pre_Institutional,1,34.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-09-19,394.8,427.8,384.5,424.1,Pre_Institutional,1,37.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-09-20,408.9,423.3,389.9,394.7,Pre_Institutional,1,36.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-09-21,398.8,412.4,393.2,408.1,Pre_Institutional,1,26.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


=== 3. Список столбцов ===
['Date', 'Close', 'High', 'Low', 'Open', 'market_era', 'halving_cycle', 'Volume_mln', 'F_Close', 'F_High', 'F_Low', 'F_Open', 'F_Volume_Contracts', 'F_Volume_USD_Mln', 'ETF_Total_Net_Flow_mln_usd', 'ETF_Cumulative_Flow_mln_usd'] 

=== 4. Типы данных (dtypes) ===
Date                            object
Close                          float64
High                           float64
Low                            float64
Open                           float64
market_era                      object
halving_cycle                    int64
Volume_mln                     float64
F_Close                        float64
F_High                         float64
F_Low                          float64
F_Open                         float64
F_Volume_Contracts             float64
F_Volume_USD_Mln               float64
ETF_Total_Net_Flow_mln_usd     float64
ETF_Cumulative_Flow_mln_usd    float64
dtype: object 

=== 5. Сводная информация (info) ===
<class 'pandas.core.frame.DataFra

In [6]:
import pandas as pd

# 1. Загружаем объединенный датасет
df_final = pd.read_csv("btc_comprehensive_analysis.csv")

# 2. Переводим столбец Date в формат datetime64[ns]
# Теперь pandas будет понимать, что это даты, а не просто текст
df_final['Date'] = pd.to_datetime(df_final['Date'])

# 3. Сортируем по дате на всякий случай (чтобы хронология была строгой)
df_final = df_final.sort_values('Date').reset_index(drop=True)

# 4. Пересохраняем файл с исправленным типом данных
final_file = "btc_comprehensive_analysis.csv"
df_final.to_csv(final_file, index=False, encoding='utf-8')

# 5. Проверяем, что тип изменился
print("✅ Тип столбца Date успешно изменен.")
print(f"Новый тип данных для Date: {df_final['Date'].dtype}")

print("\nПервые 3 строки (визуальный контроль):")
display(df_final.head(3))

✅ Тип столбца Date успешно изменен.
Новый тип данных для Date: datetime64[ns]

Первые 3 строки (визуальный контроль):


,Date,Close,High,Low,Open,market_era,halving_cycle,Volume_mln,F_Close,F_High,F_Low,F_Open,F_Volume_Contracts,F_Volume_USD_Mln,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
0,2014-09-17,457.3,468.2,452.4,465.9,Pre_Institutional,1,21.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-09-18,424.4,456.9,413.1,456.9,Pre_Institutional,1,34.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-09-19,394.8,427.8,384.5,424.1,Pre_Institutional,1,37.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1]:
import pandas as pd

# 1. Загрузка данных
df = pd.read_csv("btc_comprehensive_analysis.csv")

# 2. Приведение даты к корректному типу (на случай, если при чтении она стала объектом)
df['Date'] = pd.to_datetime(df['Date'])

# 3. Базовая информация о структуре и типах данных
print("--- ИНФОРМАЦИЯ О ДАТАФРЕЙМЕ ---")
df.info()

# 4. Проверка на наличие пропусков (NaN) по каждому столбцу
print("\n--- КОЛИЧЕСТВО ПРОПУСКОВ (NaN) ---")
print(df.isna().sum())

# 5. Проверка на полные дубликаты строк
print("\n--- КОЛИЧЕСТВО ПОЛНЫХ ДУБЛИКАТОВ ---")
print(f"Дубликатов строк: {df.duplicated().sum()}")

# 6. Статистическое описание числовых признаков (для первичной оценки разброса и выбросов)
print("\n--- СТАТИСТИЧЕСКОЕ ОПИСАНИЕ (ЧИСЛОВЫЕ) ---")
print(df.describe(include='number').round(2))

--- ИНФОРМАЦИЯ О ДАТАФРЕЙМЕ ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4175 entries, 0 to 4174
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         4175 non-null   datetime64[ns]
 1   Close                        4175 non-null   float64       
 2   High                         4175 non-null   float64       
 3   Low                          4175 non-null   float64       
 4   Open                         4175 non-null   float64       
 5   market_era                   4175 non-null   object        
 6   halving_cycle                4175 non-null   int64         
 7   Volume_mln                   4175 non-null   float64       
 8   F_Close                      2057 non-null   float64       
 9   F_High                       2057 non-null   float64       
 10  F_Low                        2057 non-null   float64       
 11  F_Open     

**Добавление расчетных признаков:** 
- Volatility_Pct - Дневная волатильность (%),
- Futures_Share_Pct - Доля фьючерсов в общем обороте (%),
- Daily_Return_Pct - Дневная доходность спота (%),
- Futures_Spread_Pct - Фьючерсный спред / премия-дисконт (%).

In [1]:
import pandas as pd
import numpy as np

# 1. Загрузка датасета (предполагаем, что файл уже в рабочей директории)
df = pd.read_csv('btc_comprehensive_analysis.csv')

# Преобразуем Date в формат datetime для корректной работы с временными рядами
df['Date'] = pd.to_datetime(df['Date'])

# 2. Расчет дневной волатильности (%)
df['Volatility_Pct'] = ((df['High'] - df['Low']) / df['Close']) * 100

# 3. Расчет дневной доходности спота (%)
df['Daily_Return_Pct'] = df['Close'].pct_change() * 100

# 4. Расчет доли фьючерсов в общем обороте (%)
df['Futures_Share_Pct'] = df['F_Volume_USD_Mln'] / (df['Volume_mln'] + df['F_Volume_USD_Mln']) * 100

# 5. Расчет фьючерсного спреда / премии-дисконта (%)
df['Futures_Spread_Pct'] = ((df['F_Close'] - df['Close']) / df['Close']) * 100

# 6. Округление новых расчетных признаков до 1 знака после запятой
# Это упрощает чтение данных и уменьшает размер итогового файла
cols_to_round = ['Volatility_Pct', 'Futures_Share_Pct', 'Daily_Return_Pct', 'Futures_Spread_Pct']
df[cols_to_round] = df[cols_to_round].round(1)

# Проверка результата: выводим первые строки с новыми столбцами
print(df[['Date'] + cols_to_round].head())

# 5. Проверка создания признаков
print("--- РАСЧЁТНЫЕ ПРИЗНАКИ СОЗДАНЫ ---")
print(f"Volatility_Pct: {df['Volatility_Pct'].notna().sum()} непустых значений")
print(f"Futures_Share_Pct: {df['Futures_Share_Pct'].notna().sum()} непустых значений (NaN до 2017)")
print(f"Daily_Return_Pct: {df['Daily_Return_Pct'].notna().sum()} непустых значений (первая строка NaN)")
print(f"Futures_Spread_Pct: {df['Futures_Spread_Pct'].notna().sum()} непустых значений (NaN до 2017)")

# 6. Быстрый осмотр первых строк с новыми признаками
display(df[['Date', 'Close', 'F_Close', 'Volatility_Pct', 'Futures_Share_Pct', 'Daily_Return_Pct', 'Futures_Spread_Pct']].head(10))

# 7. Сохранение обновленного датасета с проверкой статуса
output_file = 'btc_comprehensive_analysis_updated.csv'
try:
    # index=False нужен, чтобы pandas не сохранял лишний столбец с индексами строк
    df.to_csv(output_file, index=False)
    print(f"✅ Файл успешно сохранен: {output_file}")
except Exception as e:
    print(f"❌ Ошибка при сохранении файла: {e}")

        Date  Volatility_Pct  Futures_Share_Pct  Daily_Return_Pct  \
0 2014-09-17             3.5                NaN               NaN   
1 2014-09-18            10.3                NaN              -7.2   
2 2014-09-19            11.0                NaN              -7.0   
3 2014-09-20             8.2                NaN               3.6   
4 2014-09-21             4.8                NaN              -2.5   

   Futures_Spread_Pct  
0                 NaN  
1                 NaN  
2                 NaN  
3                 NaN  
4                 NaN  
--- РАСЧЁТНЫЕ ПРИЗНАКИ СОЗДАНЫ ---
Volatility_Pct: 4175 непустых значений
Futures_Share_Pct: 2057 непустых значений (NaN до 2017)
Daily_Return_Pct: 4174 непустых значений (первая строка NaN)
Futures_Spread_Pct: 2057 непустых значений (NaN до 2017)


,Date,Close,F_Close,Volatility_Pct,Futures_Share_Pct,Daily_Return_Pct,Futures_Spread_Pct
0,2014-09-17,457.3,NaN,3.5,NaN,NaN,NaN
1,2014-09-18,424.4,NaN,10.3,NaN,-7.2,NaN
2,2014-09-19,394.8,NaN,11.0,NaN,-7.0,NaN
3,2014-09-20,408.9,NaN,8.2,NaN,3.6,NaN
4,2014-09-21,398.8,NaN,4.8,NaN,-2.5,NaN
5,2014-09-22,402.2,NaN,2.4,NaN,0.9,NaN
6,2014-09-23,435.8,NaN,10.4,NaN,8.4,NaN
7,2014-09-24,423.2,NaN,3.5,NaN,-2.9,NaN
8,2014-09-25,411.6,NaN,3.4,NaN,-2.7,NaN
9,2014-09-26,404.4,NaN,3.7,NaN,-1.7,NaN


✅ Файл успешно сохранен: btc_comprehensive_analysis_updated.csv


In [2]:
import pandas as pd

# 1. Загрузка данных
df = pd.read_csv("btc_comprehensive_analysis_updated.csv")

# 3. Базовая информация о структуре и типах данных
print("--- ИНФОРМАЦИЯ О ДАТАФРЕЙМЕ ---")
df.info()

# 4. Проверка на наличие пропусков (NaN) по каждому столбцу
print("\n--- КОЛИЧЕСТВО ПРОПУСКОВ (NaN) ---")
print(df.isna().sum())

# 5. Проверка на полные дубликаты строк
print("\n--- КОЛИЧЕСТВО ПОЛНЫХ ДУБЛИКАТОВ ---")
print(f"Дубликатов строк: {df.duplicated().sum()}")

# 6. Статистическое описание числовых признаков (для первичной оценки разброса и выбросов)
print("\n--- СТАТИСТИЧЕСКОЕ ОПИСАНИЕ (ЧИСЛОВЫЕ) ---")
print(df.describe(include='number').round(1))

--- ИНФОРМАЦИЯ О ДАТАФРЕЙМЕ ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4175 entries, 0 to 4174
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Date                         4175 non-null   object 
 1   Close                        4175 non-null   float64
 2   High                         4175 non-null   float64
 3   Low                          4175 non-null   float64
 4   Open                         4175 non-null   float64
 5   market_era                   4175 non-null   object 
 6   halving_cycle                4175 non-null   int64  
 7   Volume_mln                   4175 non-null   float64
 8   F_Close                      2057 non-null   float64
 9   F_High                       2057 non-null   float64
 10  F_Low                        2057 non-null   float64
 11  F_Open                       2057 non-null   float64
 12  F_Volume_Contracts           2057 non-null  